Phase 1 - Imports and global styling:

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
print("Libraries loaded ✅")

Libraries loaded ✅


Phase 2 — Load & clean

## 1. Load & Clean
The raw CSV has 7,043 rows × 21 columns. `TotalCharges` arrives as `object` because
11 brand-new customers (tenure = 0) have a blank string instead of a number.
These rows are dropped: they have no billing history or tenure signal.

In [19]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

raw_shape = df.shape
print("Raw shape:", raw_shape)

# TotalCharges: text -> numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Blank TotalCharges:", df["TotalCharges"].isna().sum())   # 11

df = df.dropna(subset=["TotalCharges"]).copy()

clean_shape = df.shape
print("Clean shape:", clean_shape)                              # (7032, 21)
print("Duplicates:", df["customerID"].duplicated().sum())       # 0

Raw shape: (7043, 21)
Blank TotalCharges: 11
Clean shape: (7032, 21)
Duplicates: 0


Phase 3 — Encode the target

## 2. Target Encoding — `LabelEncoder`
`Churn` is the label, not a feature: `"No" → 0`, `"Yes" → 1`.

`LabelEncoder` is correct **here** but wrong for input features — it imposes a false
numeric ordering on non-ordinal categories (e.g. "DSL < Fiber < None" would be nonsense).
For features I use one-hot encoding instead.

In [20]:
le = LabelEncoder()
y = pd.Series(le.fit_transform(df["Churn"]), name="Churn", index=df.index)

print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Churn rate:", round(y.mean(), 4))
print(y.value_counts())

Class mapping: {'No': np.int64(0), 'Yes': np.int64(1)}
Churn rate: 0.2658
Churn
0    5163
1    1869
Name: count, dtype: int64


Mapping must show {'No': 0, 'Yes': 1} — LabelEncoder sorts alphabetically, which happens to land the right way up here. Verify it printed that, don't assume.

Phase 4 — Separate features

## 3. Feature Set
`customerID` is dropped — it is a unique key with zero predictive signal, and leaving it in
would let a model memorise rows rather than learn patterns.
`Churn` is removed because it is now the target vector.

In [21]:
X_raw = df.drop(columns=["customerID", "Churn"])

cat_cols = X_raw.select_dtypes(include="object").columns.tolist()
num_cols = X_raw.select_dtypes(include=np.number).columns.tolist()

print(f"Categorical ({len(cat_cols)}):", cat_cols)
print(f"Numerical   ({len(num_cols)}):", num_cols)
print("Feature shape before encoding:", X_raw.shape)

Categorical (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical   (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Feature shape before encoding: (7032, 19)


Phase 5 — One-hot encoding

## 4. Categorical Feature Encoding — `pd.get_dummies`
All 15 text columns are one-hot encoded. `drop_first=True` removes the first level of each
column as the reference category, avoiding perfect multicollinearity (the "dummy variable
trap") which destabilises linear models in Week 6.

**Why encode before the split?** `get_dummies` builds columns from whatever categories it
sees. If run separately on train and test, a level present in train but absent in test
produces misaligned column counts. Encoding the full feature frame once guarantees
`X_train` and `X_test` share identical columns.

In [22]:
X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True, dtype=int)

print("Feature shape after encoding:", X.shape)      # (7032, 30)
print("Non-numeric columns left:", X.select_dtypes(exclude=np.number).shape[1])   # 0
print("Any NaN:", X.isna().sum().sum())              # 0

Feature shape after encoding: (7032, 30)
Non-numeric columns left: 0
Any NaN: 0


Phase 6 — Train/test split

## 5. Train–Test Partitioning
80/20 split with `random_state=42` for reproducibility.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("y_train mean:", round(y_train.mean(), 4), "| y_test mean:", round(y_test.mean(), 4))

X_train: (5625, 30) | X_test: (1407, 30)
y_train: (5625,) | y_test: (1407,)
y_train mean: 0.2658 | y_test mean: 0.2658


Phase 7 — Standard scaling (fit on train only)

## 6. Z-Score Feature Scaling — `StandardScaler`
Three continuous columns are standardised to zero mean and unit variance:
`tenure`, `MonthlyCharges`, `TotalCharges`.

`SeniorCitizen` is **not** scaled: it is already a binary 0/1 flag, so z-scoring it
produces a meaningless range and gains nothing.

**Fit on `X_train` only, then `transform` both.** Fitting on the combined data would leak
test-set mean and standard deviation into training — the model would be tuned on
information it should not have seen, inflating Week 6 evaluation scores.

In [24]:
SCALE_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
X_test_scaled[SCALE_COLS]  = scaler.transform(X_test[SCALE_COLS])

print("Scaler fitted on train only ✅")
print(pd.DataFrame({
    "train_mean": X_train_scaled[SCALE_COLS].mean(),
    "train_std":  X_train_scaled[SCALE_COLS].std(ddof=0),
    "test_mean":  X_test_scaled[SCALE_COLS].mean(),
    "test_std":   X_test_scaled[SCALE_COLS].std(ddof=0),
}).round(4))

Scaler fitted on train only ✅
                train_mean  train_std  test_mean  test_std
tenure                -0.0        1.0     0.0176    1.0016
MonthlyCharges        -0.0        1.0     0.0167    0.9853
TotalCharges           0.0        1.0     0.0177    1.0037


Phase 8 — Verification gate

## 7. Artifact Verification
Sanity checks before export — shape alignment, no missing values, correct scaling, and
matching feature order across partitions.

In [25]:
assert X_train_scaled.shape[1] == X_test_scaled.shape[1], "column mismatch"
assert list(X_train_scaled.columns) == list(X_test_scaled.columns), "column order mismatch"
assert X_train_scaled.isna().sum().sum() == 0, "NaNs in X_train"
assert X_test_scaled.isna().sum().sum() == 0,  "NaNs in X_test"
assert len(X_train_scaled) == len(y_train), "X/y length mismatch (train)"
assert len(X_test_scaled)  == len(y_test),  "X/y length mismatch (test)"
assert set(y_train.unique()) == {0, 1}, "target is not binary"
assert 0.75 <= len(X_train_scaled) / len(X) <= 0.85, "split ratio off"

print("✅ All 8 checks passed — artifacts are consistent and ready to export.")

✅ All 8 checks passed — artifacts are consistent and ready to export.


Phase 9 — Export

## 8. Structured Artifact Export
Four CSVs exported with `index=False` so downstream scripts read a clean header row.
These are consumed directly by the Week 6 model training notebook.

In [26]:
X_train_scaled.to_csv("X_train.csv", index=False)
X_test_scaled.to_csv("X_test.csv",  index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv",   index=False)

print("Exported:")
for f in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    import os
    print(f"  {f:12s} {os.path.getsize(f)/1024:8.1f} KB")

Exported:
  X_train.csv     619.7 KB
  X_test.csv      155.5 KB
  y_train.csv      11.0 KB
  y_test.csv        2.8 KB


In [27]:
X_tr = pd.read_csv("X_train.csv")
X_te = pd.read_csv("X_test.csv")
y_tr = pd.read_csv("y_train.csv").squeeze("columns")
y_te = pd.read_csv("y_test.csv").squeeze("columns")

print("Reloaded X_train:", X_tr.shape, "| X_test:", X_te.shape)
print("Reloaded y_train:", y_tr.shape, "| y_test:", y_te.shape)
print("Columns match original:", list(X_tr.columns) == list(X_train_scaled.columns))

Reloaded X_train: (5625, 30) | X_test: (1407, 30)
Reloaded y_train: (5625,) | y_test: (1407,)
Columns match original: True


In [28]:
import joblib
joblib.dump(scaler, "scaler.pkl")
print("scaler.pkl saved — reuse this in Week 6 to transform live inputs.")

scaler.pkl saved — reuse this in Week 6 to transform live inputs.
